# MEKE backscatter runs in ACCESS-OM3

This notebook summarises results from runs using the MEKE backscatter parameterisation (with no GM/Redi parameterisation) in the dev-MC_25km_jra_iaf+wombatlite model configuration. In particular, I focus on figures that are not generated by the other evaluation figure notebooks (links below).

Name | Full name | Description | Evaluation figure notebooks
-----------|-----------|-------------|-------------|
test3v2-00532b88 | MC_25km_jra_iaf+wombatlite-test3v2-00532b88 | All runs branched off from this at 1990 | https://access-om3-paper-1.readthedocs.io/mc_25km_jra_iafwombatlite-test3v2-00532b88-2026.07.000/
bs1 | MC_25km_jra_iaf+wombatlite_bs1-mpudig-backscat1-4c21c151 | MEKE_VISCOSITY_COEFF_KU = $-0.1$ and BS_USE_SQG_STRUCT = True. No GM/Redi. Basically same parameters as GFDL were testing in OM5|https://access-om3-paper-1.readthedocs.io/mc_25km_jra_iafwombatlite_bs1-mpudig-backscat1-4c21c151-2026.07.000/
bs2 | MC_25km_jra_iaf+wombatlite_bs2-mpudig-backscat2-0578cc36 | Same as bs1 except MEKE_VISCOSITY_COEFF_KU = $-0.2$| https://access-om3-paper-1.readthedocs.io/mc_25km_jra_iafwombatlite_bs2-mpudig-backscat2-0578cc36-2026.07.000/

Note: I also had to change MAX_DELTA_SRESTORE from 1.0 to 10.0 -- see https://github.com/ACCESS-NRI/access-om3-configs/issues/1235

In [ ]:
#This cell must be in all notebooks!
#It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

### USER EDIT start
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test3v2-00532b88/datastore.json"
esm_file_bs1 = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite_bs1-mpudig-backscat1-4c21c151/datastore.json"
esm_file_bs2 = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite_bs2-mpudig-backscat2-0578cc36/datastore.json"
dpi=300
### USER EDIT stop

import os
from matplotlib import rcParams
%matplotlib inline
rcParams["figure.dpi"]= dpi

plotfolder=f"/g/data/{os.environ['PROJECT']}/{os.environ['USER']}/access-om3-paper-figs/"
#os.makedirs(plotfolder, exist_ok=True)

 # a similar cell under this means it's being run in batch
print("ESM datastore path: ",esm_file)
print("Plot folder path: ",plotfolder)

In [ ]:
IAF = esm_file.find('iaf') > 0
IAF

In [ ]:
import xarray as xr
import cf_xarray
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client
import cftime
import cmocean as cm
import numpy as np
import cartopy.feature as feature
import matplotlib.path as mpath
from tqdm.notebook import tqdm

In [ ]:
client = Client(threads_per_worker=1)
print(client.dashboard_link)

### Open the intake-esm datastore

In [ ]:
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

datastore_bs1 = intake.open_esm_datastore(
    esm_file_bs1,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

datastore_bs2 = intake.open_esm_datastore(
    esm_file_bs2,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

### What ocean variables are available at monthly frequency?

In [ ]:
def available_variables(datastore):
    """Return a pandas dataframe summarising the variables in a datastore"""
    variable_columns = [col for col in datastore.df.columns if "variable" in col]
    return (
        datastore.df[variable_columns]
        .explode(variable_columns)
        .drop_duplicates()
        .set_index("variable")
        .sort_index()
    )

In [ ]:
datastore_bs1_filtered = datastore_bs1.search(realm="ocean", frequency="1mon", variable="umo")

available_variables(datastore_bs1_filtered)

## Get coordinates

In [ ]:
try:
# preferred source, if available (no processor masking holes)
# path is a workaround - see https://forum.access-hive.org.au/t/issues-in-loading-ht-in-latest-conds-envs/6278
    path = datastore.search(variable=["geolat", "geolon"], filename="access-om3.mom6.geometry.nc").df.loc[0, 'path']
    coords = datastore.search(variable=["geolat", "geolon"], path=path).to_dask().compute()
    coords = coords.rename({ "lonh": "xh", "lath": "yh" })
except:
# these contain NaNs in processor masks, so set NaNs to zero so they can be used for plotting
# path is a workaround - see https://forum.access-hive.org.au/t/issues-in-loading-ht-in-latest-conds-envs/6278
    path = datastore.search(variable=["geolat", "geolon"], filename="access-om3.mom6.static.nc").df.loc[0, 'path']
    coords = datastore.search(variable=["geolat", "geolon"], path=path).to_dask().compute()
    coords = coords.fillna(0.0)

## Look at geography of subgrid MEKE and surface negative viscosity

MEKE indicates how much energy is available within a local subgrid KE reservoir to be backscattered to resolved scales. The negative viscosity is modulated by the magnitude of MEKE, a geographically-varying length scale, and a geographically-varying exponential vertical structure.

In [ ]:
start_time = '2019-01-01'
end_time = '2019-12-31'

variable = 'MEKE'

meke_bs1 = datastore_bs1.search(variable = variable, frequency = '1mon').to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks = {'time': -1},
        decode_timedelta = True
    )
)[variable].sel(time = slice(start_time, end_time))

meke_bs2 = datastore_bs2.search(variable = variable, frequency = '1mon').to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks = {'time': -1},
        decode_timedelta = True
    )
)[variable].sel(time = slice(start_time, end_time))

In [ ]:
%%time
meke_bs1_tav = meke_bs1.mean('time').load()

In [ ]:
%%time
meke_bs2_tav = meke_bs2.mean('time').load()

In [ ]:
proj = ccrs.PlateCarree(central_longitude = -100)
fig, axs = plt.subplots(1,2, figsize=(10,5), subplot_kw=dict(projection=proj))
vmax = 0.1
vmin = 0.
cmap = cm.cm.thermal

ax = axs[0]
p1 = meke_bs1_tav.assign_coords(coords).plot(x="geolon", y="geolat", ax = ax, vmin = vmin, vmax = vmax, cmap = cmap, add_colorbar = False, transform=ccrs.PlateCarree())
ax.coastlines()
ax.set_title('bs1')

ax = axs[1]
p1 = meke_bs2_tav.assign_coords(coords).plot(x="geolon", y="geolat", ax = ax, vmin = vmin, vmax = vmax, cmap = cmap, add_colorbar = False, transform=ccrs.PlateCarree())
ax.coastlines()
ax.set_title('bs2')

ax_c = plt.axes([0.92,0.3,0.01,0.4])
pc = plt.colorbar(p1,ax_c)
plt.xlabel(r'MEKE [m$^2$ / s$^2$]')

In [ ]:
start_time = '2019-01-01'
end_time = '2019-12-31'

variable = 'difmxylo'
zl = 0.

khh_bs1 = datastore_bs1.search(variable = variable, frequency = '1mon').to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks = {'time': -1},
        decode_timedelta = True
    )
)[variable].sel(z_l = 0, method = 'nearest').sel(time = slice(start_time, end_time))

khh_bs2 = datastore_bs2.search(variable = variable, frequency = '1mon').to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks = {'time': -1},
        decode_timedelta = True
    )
)[variable].sel(z_l = 0, method = 'nearest').sel(time = slice(start_time, end_time))

In [ ]:
%%time
khh_bs1_tav = khh_bs1.mean('time').load()

In [ ]:
%%time
khh_bs2_tav = khh_bs2.mean('time').load()

In [ ]:
proj = ccrs.PlateCarree(central_longitude = -100)
fig, axs = plt.subplots(1,2, figsize=(10,5), subplot_kw=dict(projection=proj))
vmax = 0.
vmin = -800.
cmap = cm.cm.algae

ax = axs[0]
p1 = khh_bs1_tav.assign_coords(coords).plot(x="geolon", y="geolat", ax = ax, vmin = vmin, vmax = vmax, cmap = cmap, add_colorbar = False, transform=ccrs.PlateCarree())
ax.coastlines()
ax.set_title('bs1')

ax = axs[1]
p1 = khh_bs2_tav.assign_coords(coords).plot(x="geolon", y="geolat", ax = ax, vmin = vmin, vmax = vmax, cmap = cmap, add_colorbar = False, transform=ccrs.PlateCarree())
ax.coastlines()
ax.set_title('bs2')

ax_c = plt.axes([0.92,0.3,0.01,0.4])
pc = plt.colorbar(p1,ax_c)
plt.xlabel(r'$\nu_2$ [m$^2$ / s]')

In [ ]:
plt.figure(figsize = (8, 4))

khh_bs1_tav.mean('xh').plot(label = 'bs1')
khh_bs2_tav.mean('xh').plot(label = 'bs2')
plt.title('Zonal mean negative viscosity')
plt.xlabel('Latitude')
plt.ylabel('[m$^{2}$ / s]')
plt.legend()
plt.ylim(None, 0)

### Sea surface speed

In [ ]:
start_time = '2009-12-31'
end_time = '2019-12-31'

variable = 'speed'

speed = datastore.search(variable = variable, frequency = '1mon').to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks = {'time': -1},
        decode_timedelta = True
    )
)[variable].sel(time = slice(start_time, end_time))

speed_bs1 = datastore_bs1.search(variable = variable, frequency = '1mon').to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks = {'time': -1},
        decode_timedelta = True
    )
)[variable].sel(time = slice(start_time, end_time))

speed_bs2 = datastore_bs2.search(variable = variable, frequency = '1mon').to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks = {'time': -1},
        decode_timedelta = True
    )
)[variable].sel(time = slice(start_time, end_time))

In [ ]:
speed_snap = speed.isel(time = -1).load()
speed_bs1_snap = speed_bs1.isel(time = -1).load()
speed_bs2_snap = speed_bs2.isel(time = -1).load()

Monthly snapshot of sea surface speed globally

In [ ]:
proj = ccrs.PlateCarree(central_longitude = -100)
fig, axs = plt.subplots(2,2, figsize=(10,12), subplot_kw=dict(projection=proj))
vmin = 0.1
vmax = 0.6
cmap = cm.cm.rain

axs[0, 1].axis('off')

ax = axs[0, 0]
p1 = speed_snap.assign_coords(coords).plot(x="geolon", y="geolat", ax = ax, vmin = vmin, vmax = vmax, cmap = cmap, add_colorbar = False, transform=ccrs.PlateCarree())
ax.coastlines()
ax.set_title('test3v2-00532b88')

ax = axs[1, 0]
p1 = speed_bs1_snap.assign_coords(coords).plot(x="geolon", y="geolat", ax = ax, vmin = vmin, vmax = vmax, cmap = cmap, add_colorbar = False, transform=ccrs.PlateCarree())
ax.coastlines()
ax.set_title('bs1')

ax = axs[1, 1]
p1 = speed_bs2_snap.assign_coords(coords).plot(x="geolon", y="geolat", ax = ax, vmin = vmin, vmax = vmax, cmap = cmap, add_colorbar = False, transform=ccrs.PlateCarree())
ax.coastlines()
ax.set_title('bs2')

fig.subplots_adjust(hspace = -0.7)

ax_c = plt.axes([0.92,0.4,0.01,0.15])
pc = plt.colorbar(p1, ax_c)
pc.set_label(r'Surface speed [m/s]')

Plot 10-yr time average (2010--2019) surface currents in same WBC regions as in GM-Testing notebook -- https://github.com/ACCESS-Community-Hub/access-om3-paper-1/blob/main/notebooks/GM-Testing-in-ACCESS-OM3.ipynb

In [ ]:
%%time
speed_tav = speed.mean('time').load()

In [ ]:
%%time
speed_bs1_tav = speed_bs1.mean('time').load()

In [ ]:
%%time
speed_bs2_tav = speed_bs2.mean('time').load()

In [ ]:
fig = plt.figure(figsize = (12, 12)) 

cmap = cm.cm.rain
vmax = 0.6
vmin = 0.2

plt.subplot(4,3,1)
speed_tav.sel(xh=slice(-85,-35)).sel(yh=slice(25,55)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('lat')
plt.xlabel('')
plt.title('test3v2-00532b88')

plt.subplot(4,3,2)
speed_bs1_tav.sel(xh=slice(-85,-35)).sel(yh=slice(25,55)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('')
plt.xlabel('')
plt.title('bs1')
plt.tick_params(axis = 'y', labelleft = False)

plt.subplot(4,3,3)
speed_bs2_tav.sel(xh=slice(-85,-35)).sel(yh=slice(25,55)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('')
plt.xlabel('')
plt.title('bs2')
plt.tick_params(axis = 'y', labelleft = False)

plt.subplot(4,3,4)
speed_tav.sel(xh=slice(-240,-190)).sel(yh=slice(25,55)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('lat')
plt.xlabel('')
#plt.title('test3v2-00532b88')

plt.subplot(4,3,5)
speed_bs1_tav.sel(xh=slice(-240,-190)).sel(yh=slice(25,55)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('')
plt.xlabel('')
#plt.title('bs1')
plt.tick_params(axis = 'y', labelleft = False)

plt.subplot(4,3,6)
speed_bs2_tav.sel(xh=slice(-240,-190)).sel(yh=slice(25,55)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('')
plt.xlabel('')
#plt.title('bs2')
plt.tick_params(axis = 'y', labelleft = False)

plt.subplot(4,3,7)
speed_tav.sel(xh=slice(-220,-180)).sel(yh=slice(-55,-25)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('lat')
plt.xlabel('')
#plt.title('test3v2-00532b88')

plt.subplot(4,3,8)
speed_bs1_tav.sel(xh=slice(-220,-180)).sel(yh=slice(-55,-25)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('')
plt.xlabel('')
#plt.title('bs1')
plt.tick_params(axis = 'y', labelleft = False)

plt.subplot(4,3,9)
speed_bs2_tav.sel(xh=slice(-220,-180)).sel(yh=slice(-55,-25)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('')
plt.xlabel('')
#plt.title('bs2')
plt.tick_params(axis = 'y', labelleft = False)

plt.subplot(4,3,10)
speed_tav.sel(xh=slice(10,50)).sel(yh=slice(-55,-25)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('lat')
plt.xlabel('lon')
#plt.title('test3v2-00532b88')

plt.subplot(4,3,11)
speed_bs1_tav.sel(xh=slice(10,50)).sel(yh=slice(-55,-25)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('')
plt.xlabel('lon')
#plt.title('bs1')
plt.tick_params(axis = 'y', labelleft = False)

plt.subplot(4,3,12)
im = speed_bs2_tav.sel(xh=slice(10,50)).sel(yh=slice(-55,-25)).plot(vmax = vmax, vmin = vmin, cmap = cmap, add_colorbar = False)
plt.ylabel('')
plt.xlabel('lon')
#plt.title('bs2')
plt.tick_params(axis = 'y', labelleft = False)

cbar_ax = fig.add_axes([0.15, 0.02, 0.725, 0.015])
cbar = fig.colorbar(im, cax = cbar_ax, orientation = 'horizontal')
cbar.set_label(r'sea surface speed [m s$^{-1}$]')

plt.subplots_adjust(hspace = 0.2, wspace = 0.15, bottom = 0.08)

### Depth-integrated KE

Look at monthly average of depth averaged KE

In [ ]:
start_time = '2019-01-01'
end_time = '2019-01-31'

# Kinetic energy
variable = 'KE'

# control
KE = datastore.search(
    variable = variable,
    frequency = '1mon').to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat = 'override',
        data_vars = 'minimal',
        coords = 'minimal',
    ),
    xarray_open_kwargs = dict(
        chunks = {'yh' : -1, 'xh': -1}, # Good for spatial operations, but not temporal
        decode_timedelta = True
    ),
)[variable].sel(time = slice(start_time, end_time)).isel(time = 0)

# bs1
KE_bs1 = datastore_bs1.search(
    variable = variable,
    frequency = '1mon').to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat = 'override',
        data_vars = 'minimal',
        coords = 'minimal',
    ),
    xarray_open_kwargs = dict(
        chunks = {'yh' : -1, 'xh': -1}, # Good for spatial operations, but not temporal
        decode_timedelta = True
    ),
)[variable].sel(time = slice(start_time, end_time)).isel(time = 0)

# bs2
KE_bs2 = datastore_bs2.search(
    variable = variable,
    frequency = '1mon').to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat = 'override',
        data_vars = 'minimal',
        coords = 'minimal',
    ),
    xarray_open_kwargs = dict(
        chunks = {'yh' : -1, 'xh': -1}, # Good for spatial operations, but not temporal
        decode_timedelta = True
    ),
)[variable].sel(time = slice(start_time, end_time)).isel(time = 0)

# Thickness
variable = 'thkcello'

# control
thkcello = datastore.search(
            variable = variable,
            frequency = '1mon',
            file_id = 'ocean.1mon.nv:2.xh:1440.yh:1152.z_i:76.z_l:75').to_dask(
                xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
                    compat = 'override',
                    data_vars = 'minimal',
                    coords = 'minimal',
                ),
                xarray_open_kwargs = dict(
                    chunks = {'yh' : -1, 'xh': -1}, # Good for spatial operations, but not temporal
                    decode_timedelta = True
                ),
)[variable].sel(time = slice(start_time, end_time)).isel(time = 0)

thkcello_bs1 = datastore_bs1.search(
            variable = variable,
            frequency = '1mon',
            file_id = 'ocean.1mon.nv:2.xh:1440.yh:1152.z_i:76.z_l:75').to_dask(
                xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
                    compat = 'override',
                    data_vars = 'minimal',
                    coords = 'minimal',
                ),
                xarray_open_kwargs = dict(
                    chunks = {'yh' : -1, 'xh': -1}, # Good for spatial operations, but not temporal
                    decode_timedelta = True
                ),
)[variable].sel(time = slice(start_time, end_time)).isel(time = 0)

thkcello_bs2 = datastore_bs2.search(
            variable = variable,
            frequency = '1mon',
            file_id = 'ocean.1mon.nv:2.xh:1440.yh:1152.z_i:76.z_l:75').to_dask(
                xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
                    compat = 'override',
                    data_vars = 'minimal',
                    coords = 'minimal',
                ),
                xarray_open_kwargs = dict(
                    chunks = {'yh' : -1, 'xh': -1}, # Good for spatial operations, but not temporal
                    decode_timedelta = True
                ),
)[variable].sel(time = slice(start_time, end_time)).isel(time = 0)

In [ ]:
%%time
KE_zav = ((KE * thkcello).sum('z_l') / thkcello.sum('z_l')).load()

In [ ]:
%%time
KE_bs1_zav = ((KE_bs1 * thkcello_bs1).sum('z_l') / thkcello_bs1.sum('z_l')).load()

In [ ]:
%%time
KE_bs2_zav = ((KE_bs2 * thkcello_bs2).sum('z_l') / thkcello_bs2.sum('z_l')).load()

In [ ]:
proj = ccrs.PlateCarree(central_longitude = -100)
fig, axs = plt.subplots(2,2, figsize=(10,12), subplot_kw=dict(projection=proj))
vmin = -4
vmax = -1
cmap = cm.cm.rain

axs[0, 1].axis('off')

ax = axs[0, 0]
p1 = np.log10(KE_zav).assign_coords(coords).plot(x="geolon", y="geolat", ax = ax, vmin = vmin, vmax = vmax, cmap = cmap, add_colorbar = False, transform=ccrs.PlateCarree())
ax.coastlines()
ax.set_title('test3v2-00532b88')

ax = axs[1, 0]
p1 = np.log10(KE_bs1_zav).assign_coords(coords).plot(x="geolon", y="geolat", ax = ax, vmin = vmin, vmax = vmax, cmap = cmap, add_colorbar = False, transform=ccrs.PlateCarree())
ax.coastlines()
ax.set_title('bs1')

ax = axs[1, 1]
p1 = np.log10(KE_bs2_zav).assign_coords(coords).plot(x="geolon", y="geolat", ax = ax, vmin = vmin, vmax = vmax, cmap = cmap, add_colorbar = False, transform=ccrs.PlateCarree())
ax.coastlines()
ax.set_title('bs2')

fig.subplots_adjust(hspace = -0.7)

ax_c = plt.axes([0.92,0.4,0.01,0.15])
pc = plt.colorbar(p1,ax_c)
pc.set_label(r'Depth averaged EKE [log$_{10}$ m$^2$/s$^2$]')

As there is a lot of increased eddy activity in the Southern Ocean, it's useful to make a circumpolar plot.

In [ ]:
def select_region(ds):
    ycoord = 'yh' if 'yh' in ds.dims else 'yq'
    return ds.sel({ycoord: slice(None, -50)})

deptho = datastore_bs1.search(variable = 'deptho')
deptho = deptho.search(path = deptho.df.loc[0, 'path']).to_dask(xarray_open_kwargs = {'chunks':'auto'})

land_mask = xr.where(np.isnan(deptho['deptho']), 1, np.nan)
land_mask = land_mask.rename('land_mask').sel(yh = slice(None, -49))

land_50m = feature.NaturalEarthFeature('physical', 'land', '50m',
                                        edgecolor = 'black',
                                        facecolor = 'gray',
                                        linewidth = 0.2)
def circumpolar_map():
    fig = plt.figure(figsize = (12, 8), dpi = 150)
    ax = plt.axes(projection = ccrs.SouthPolarStereo())
    ax.set_extent([-180, 180, -80, -50], crs = ccrs.PlateCarree())
    ax.set_facecolor('lightgrey')
    # Map the plot boundaries to a circle
    theta = np.linspace(0, 2 * np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform = ax.transAxes)
    land_mask.plot.contourf(ax = ax, colors = 'lightgrey', add_colorbar = False, 
                            zorder = 2, transform = ccrs.PlateCarree())
    return fig, ax

In [ ]:
vmin = 0.
vmax = 0.05
cmap = cm.cm.rain

ds = select_region(KE_zav)
fig, axs = circumpolar_map()
time = ds.time.dt.strftime('%Y-%m-%d').values.item()
ds.plot(ax = axs, vmin = vmin, vmax = vmax, cmap = cmap, transform = ccrs.PlateCarree(),
         cbar_kwargs = {'label':'m$^2$ s$^{-2}$', 'shrink' : 0.6});
axs.set_title(f'Depth averaged KE, test3v2-00532b88, {time}');

ds = select_region(KE_bs1_zav)
fig, axs = circumpolar_map()
time = ds.time.dt.strftime('%Y-%m-%d').values.item()
ds.plot(ax = axs, vmin = vmin, vmax = vmax, cmap = cmap, transform = ccrs.PlateCarree(),
         cbar_kwargs = {'label':'m$^2$ s$^{-2}$', 'shrink' : 0.6});
axs.set_title(f'Depth averaged KE, bs1, {time}');

ds = select_region(KE_bs2_zav)
fig, axs = circumpolar_map()
time = ds.time.dt.strftime('%Y-%m-%d').values.item()
ds.plot(ax = axs, vmin = vmin, vmax = vmax, cmap = cmap, transform = ccrs.PlateCarree(),
         cbar_kwargs = {'label':'m$^2$ s$^{-2}$', 'shrink' : 0.6});
axs.set_title(f'Depth averaged KE, bs2, {time}');

### Bottom age in Southern Ocean

The bs2 run has had a massive effect on ice around West Antarctica... Bottom age acts as a proxy for this.

This region is also where SSTs are substantially higher (see evaluation figure plot - https://access-om3-paper-1.readthedocs.io/mc_25km_jra_iafwombatlite_bs2-mpudig-backscat2-0578cc36-2026.07.000/experiments/MC_25km_jra_iaf%2Bwombatlite_bs2-mpudig-backscat2-0578cc36/SST/). Zonal mean plots of T and S also indicate lots of cold, fresh water at depth at these latitudes (done below).

In [ ]:
time_start = '2019-12-01'
time_end = '2019-12-31'

variable = "agessc"

age = datastore.search(variable=variable, frequency="1mon").to_dask(
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        # chunks={"time": -1}, # Good for temporal operations, but not spatial
        decode_timedelta=True
    )
)[variable].sel(yh = slice(-90, -50)).sel(time = slice(time_start, time_end)).isel(time = 0).load()

age_bs1 = datastore_bs1.search(variable=variable, frequency="1mon").to_dask(
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        # chunks={"time": -1}, # Good for temporal operations, but not spatial
        decode_timedelta=True
    )
)[variable].sel(yh = slice(-90, -50)).sel(time = slice(time_start, time_end)).isel(time = 0).load()

age_bs2 = datastore_bs2.search(variable=variable, frequency="1mon").to_dask(
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        # chunks={"time": -1}, # Good for temporal operations, but not spatial
        decode_timedelta=True
    )
)[variable].sel(yh = slice(-90, -50)).sel(time = slice(time_start, time_end)).isel(time = 0).load()

In [ ]:
# Get bottom tracer
depth_array = age * 0 + age.z_l
max_depth   = depth_array.max(dim = 'z_l', skipna = True)
age_bot = age.where(depth_array.z_l >= max_depth)
age_bot = age_bot.sum(dim='z_l').load()

In [ ]:
# Get bottom tracer
depth_array = age_bs1 * 0 + age_bs1.z_l
max_depth   = depth_array.max(dim = 'z_l', skipna = True)
age_bs1_bot = age_bs1.where(depth_array.z_l >= max_depth)
age_bs1_bot = age_bs1_bot.sum(dim='z_l').load()

In [ ]:
# Get bottom tracer
depth_array = age_bs2 * 0 + age_bs2.z_l
max_depth   = depth_array.max(dim = 'z_l', skipna = True)
age_bs2_bot = age_bs2.where(depth_array.z_l >= max_depth)
age_bs2_bot = age_bs2_bot.sum(dim='z_l').load()

In [ ]:
# Normalise age tracer by max value
age_bot_normalised = age_bot / age_bot.max()
age_bs1_bot_normalised = age_bs1_bot / age_bs1_bot.max()
age_bs2_bot_normalised = age_bs2_bot / age_bs2_bot.max()

In [ ]:
variable = 'deptho'
var_search = datastore.search(variable = variable)
bathymetry = var_search.search(path=var_search.df.path[0]).to_dask(
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        # chunks={"time": -1}, # Good for temporal operations, but not spatial
        decode_timedelta=False
    )
)[variable].sel(yh = slice(-90, -50)).load()
land = xr.where(np.isnan(bathymetry.rename('land')), 1, np.nan)

In [ ]:
vmax = 1.
vmin = 0.
cmap = cm.cm.rain_r
yh_min = -52
yh_max = -82

fig, axs = plt.subplots(figsize = (15, 10), ncols = 1, nrows = 3, sharex = True, dpi = 150)

ax = axs[0]
age_normalised.sel(yh = slice(yh_max, yh_min)).plot(ax = ax, vmax = vmax, vmin = vmin, cmap = cmap, cbar_kwargs = {'label': "Normalised bottom age [yr]"})
land.plot.contourf(ax = ax, colors = 'darkgrey', add_colorbar = False)
ax.set_ylabel('Latitude [°N]')
ax.set_xlabel('')
ax.set_title('test3v2-00532b88')

ax = axs[1]
age_bs1_bot_normalised.sel(yh = slice(yh_max, yh_min)).plot(ax = ax, vmax = vmax, vmin = vmin, cmap = cmap, cbar_kwargs = {'label': "Normalised bottom age [yr]"})
land.plot.contourf(ax = ax, colors = 'darkgrey', add_colorbar = False)
ax.set_ylabel('Latitude [°N]')
ax.set_xlabel('')
ax.set_title('bs1')

ax = axs[2]
age_bs2_bot_normalised.sel(yh = slice(yh_max, yh_min)).plot(ax = ax, vmax = vmax, vmin = vmin, cmap = cmap, cbar_kwargs = {'label': "Normalised bottom age [yr]"})
land.plot.contourf(ax = ax, colors = 'darkgrey', add_colorbar = False)
ax.set_ylabel('Latitude [°N]')
ax.set_xlabel('Longitude [°E]')
ax.set_title('bs2')

### Zonal means: age, T, S

In [ ]:
catalogs = [esm_file,
            esm_file_bs1,
            esm_file_bs2,
           ]

datastores = { os.path.normpath(c).split(os.sep)[-2]:
               intake.open_esm_datastore(c,
                                         columns_with_iterables=[
                                            "variable",
                                            "variable_long_name",
                                            "variable_standard_name",
                                            "variable_cell_methods",
                                            "variable_units"]
                                        )
              for c in catalogs }

First look at zonal mean age averaged over final simulation year.

In [ ]:
varnames = ['agessc']
time_start = '2019-01-01'
time_end = '2019-12-31'

def drop_spatial_coords(ds):
    coords_to_drop = [c for c in ['yh', 'xh'] if c in ds.coords]
    return ds.drop_vars(coords_to_drop)

om3vars = {vname: {expt: ds.search(variable=vname).to_dask(
                xarray_open_kwargs=dict(
                    chunks={"xh": -1, "yh": -1, "z_l": -1},
                    decode_timedelta=True,
                ),
                preprocess=drop_spatial_coords,
            )[vname]
            for expt, ds in datastores.items()}
    for vname in varnames}

# reattach yh and xh
om3vars = {
    vname: {
        expt: da.assign_coords(
            xh = coords['xh'],
            yh = coords['yh'],
        )
        for expt, da in vdatadict.items()
    }
    for vname, vdatadict in om3vars.items()
}

lasttime = []
for vname, d in om3vars.items():
    for expt, da in d.items():
        lasttime.append(da.time.values[-1])


timerange = slice(time_start,
                  time_end)

regions = { # [minx, maxx, miny, maxy], using model longitude range (-280 to 80)
    "Global": [-280, 80, -90, 90],
    # "Arctic": [-280, 80, 65, 90],
    # "Southern Ocean": [-280, 80, -82, -63],
    # "ACC": [-280, 80, -63, -45],
    # "Southern Pacific": [-210, -70, -45, -20],
    # "Tropical Pacific": [-240, -100, -20, 20],
    # "North Pacific": [-240, -100, 20, 65],
    # "South Atlantic": [-60, 20, -45, -20],
    # "Tropical Atlantic": [-70, 20, -20, 20],
    # "North Atlantic": [-100, 0, 20, 65],
    # "Indian": [30, 120, -45, 20],
    # "Aegean Sea": [18, 27.5, 34, 44],
    # "Black Sea": [27.5, 43, 40.5, 48],
    # "Baltic Sea": [13, 30, 53, 58],
    # "Mediterranean Sea": [0, 35, 31, 41],
    # "Red Sea": [33, 44, 12, 29],
    # "Persian Gulf": [47, 56, 24, 31],
    # "White Sea": [31, 41, 63, 68],
}
regions = {k: dict(zip(["minx", "maxx", "miny", "maxy"], v)) for k, v in regions.items()}  # convert to dicts

for r, d in regions.items():
    for k, x in d.items():
        if k in ["minx", "maxx"] and x != max(-280, min(x, 80)):
            raise ValueError(f"{r} {k} = {x} is outside the range -280 to 80")
        if k in ["miny", "maxy"] and x > 65:
            print(f"{r} {k} changed from {x} to 65 to omit tripolar region")
            d[k] = 65

In [ ]:
%%time
om3vars_mean = {
    region:
    {
        vname: {
                expt: da.sel(yh=slice(b['miny'], b['maxy']))\
                        .sel(xh=slice(b['minx'], b['maxx']))\
                        .sel(time=timerange)\
                        .mean('xh').mean('time').load()
                for expt, da in tqdm(vdatadict.items(), desc=f'      {vname} experiments', )
              }
        for vname, vdatadict in tqdm(om3vars.items(), desc=f'   {region} variables')
    }
    for region, b in tqdm(regions.items(), desc='regions')
}

In [ ]:
var = 'agessc'
ds = om3vars_mean['Global'][var]

da = ds[list(ds)[0]]
da_bs1 = ds[list(ds)[1]]
da_bs2 = ds[list(ds)[2]]

dpi = 300
fontsize = 12
fig, axs = plt.subplots(2, 3, figsize=(25, 11), sharex=True, sharey=True)
fig.suptitle(f'Global zonal mean ideal age, {time_start} - {time_end} mean', fontsize=fontsize + 2)

vmax = 60
vmin = 0.
cmap = cm.cm.rain_r

ax = axs[0, 0]
ax.set_facecolor('gray')
p = da.plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
da.plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['k'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('test3v2-00532b88', fontsize=fontsize)
ax.set_xlabel('Latitude [°N]', fontsize=fontsize)
ax.set_ylabel('Depth [m]', fontsize=fontsize)

ax = axs[0, 1]
ax.set_facecolor('gray')
p = da_bs1.plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
da_bs1.plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['k'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs1', fontsize=fontsize)
ax.set_xlabel('', fontsize=fontsize)
ax.set_ylabel('')

axs[1, 0].axis('off')

ax = axs[1, 1]
ax.set_facecolor('gray')
p = da_bs2.plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
da_bs2.plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['k'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs2', fontsize=fontsize)
ax.set_xlabel('Latitude [°N]', fontsize=fontsize)
ax.set_ylabel('')

pos0 = axs[1, 0].get_position()
pos1 = axs[1, 1].get_position()
cax = fig.add_axes([pos0.x0, pos0.y0 - 0.08, pos1.x1 - pos0.x0, 0.03])
fig.colorbar(p, cax=cax, orientation='horizontal', label='Ideal age [yr]', extend='both')

ax = axs[0, 2]
ax.set_facecolor('gray')
vmax = 10
vmin = -vmax
cmap = cm.cm.balance

p = (da_bs1 - da).plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
(da_bs1 - da).plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['gray'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs1 minus test3v2-00532b88', fontsize=fontsize)
ax.set_xlabel('', fontsize=fontsize)
ax.set_ylabel('')

pos = ax.get_position()
cax = fig.add_axes([pos.x1 + 0.1, pos.y0 + 0.05, 0.015, pos.y1 - pos.y0])
fig.colorbar(p, cax = cax, orientation='vertical', label='Ideal age [yr]', extend='both')

ax = axs[1, 2]
ax.set_facecolor('gray')
vmax = 50
vmin = -vmax
cmap = cm.cm.balance

p = (da_bs2 - da).plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
(da_bs2 - da).plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['gray'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs2 minus test3v2-00532b88', fontsize=fontsize)
ax.set_xlabel('Latitude [°N]', fontsize=fontsize)
ax.set_ylabel('')

pos = ax.get_position()
cax = fig.add_axes([pos.x1 + 0.1, pos.y0 + 0.05, 0.015, pos.y1 - pos.y0])
fig.colorbar(p, cax = cax, orientation='vertical', label='Ideal age [yr]', extend='neither')

for ax in fig.get_axes():
    ax.label_outer()
plt.tight_layout(rect=[0, 0.1, 1, 1])

Next look at zonal mean T and S averaged over 10 year period.

In [ ]:
varnames = ['thetao', 'so']
time_start = '2009-12-31'
time_end = '2019-12-31'

def drop_spatial_coords(ds):
    coords_to_drop = [c for c in ['yh', 'xh'] if c in ds.coords]
    return ds.drop_vars(coords_to_drop)

om3vars = {vname: {expt: ds.search(variable=vname).to_dask(
                xarray_open_kwargs=dict(
                    chunks={"xh": -1, "yh": -1, "z_l": -1},
                    decode_timedelta=True,
                ),
                preprocess=drop_spatial_coords,
            )[vname]
            for expt, ds in datastores.items()}
    for vname in varnames}

# reattach yh and xh
om3vars = {
    vname: {
        expt: da.assign_coords(
            xh = coords['xh'],
            yh = coords['yh'],
        )
        for expt, da in vdatadict.items()
    }
    for vname, vdatadict in om3vars.items()
}

lasttime = []
for vname, d in om3vars.items():
    for expt, da in d.items():
        lasttime.append(da.time.values[-1])


timerange = slice(time_start,
                  time_end)

regions = { # [minx, maxx, miny, maxy], using model longitude range (-280 to 80)
    "Global": [-280, 80, -90, 90],
    # "Arctic": [-280, 80, 65, 90],
    # "Southern Ocean": [-280, 80, -82, -63],
    # "ACC": [-280, 80, -63, -45],
    # "Southern Pacific": [-210, -70, -45, -20],
    # "Tropical Pacific": [-240, -100, -20, 20],
    # "North Pacific": [-240, -100, 20, 65],
    # "South Atlantic": [-60, 20, -45, -20],
    # "Tropical Atlantic": [-70, 20, -20, 20],
    # "North Atlantic": [-100, 0, 20, 65],
    # "Indian": [30, 120, -45, 20],
    # "Aegean Sea": [18, 27.5, 34, 44],
    # "Black Sea": [27.5, 43, 40.5, 48],
    # "Baltic Sea": [13, 30, 53, 58],
    # "Mediterranean Sea": [0, 35, 31, 41],
    # "Red Sea": [33, 44, 12, 29],
    # "Persian Gulf": [47, 56, 24, 31],
    # "White Sea": [31, 41, 63, 68],
}
regions = {k: dict(zip(["minx", "maxx", "miny", "maxy"], v)) for k, v in regions.items()}  # convert to dicts

for r, d in regions.items():
    for k, x in d.items():
        if k in ["minx", "maxx"] and x != max(-280, min(x, 80)):
            raise ValueError(f"{r} {k} = {x} is outside the range -280 to 80")
        if k in ["miny", "maxy"] and x > 65:
            print(f"{r} {k} changed from {x} to 65 to omit tripolar region")
            d[k] = 65

In [ ]:
%%time
om3vars_mean = {
    region:
    {
        vname: {
                expt: da.sel(yh=slice(b['miny'], b['maxy']))\
                        .sel(xh=slice(b['minx'], b['maxx']))\
                        .sel(time=timerange)\
                        .mean('xh').mean('time').load()
                for expt, da in tqdm(vdatadict.items(), desc=f'      {vname} experiments', )
              }
        for vname, vdatadict in tqdm(om3vars.items(), desc=f'   {region} variables')
    }
    for region, b in tqdm(regions.items(), desc='regions')
}

In [ ]:
var = 'thetao'
ds = om3vars_mean['Global'][var]

da = ds[list(ds)[0]]
da_bs1 = ds[list(ds)[1]]
da_bs2 = ds[list(ds)[2]]

dpi = 300
fontsize = 12
fig, axs = plt.subplots(2, 3, figsize=(25, 11), sharex=True, sharey=True)
fig.suptitle(f'Global zonal mean temperature, {time_start} - {time_end} mean', fontsize=fontsize + 2)

vmax = 22
vmin = -1
cmap = cm.cm.thermal

ax = axs[0, 0]
ax.set_facecolor('gray')
p = da.plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
da.plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['k'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('test3v2-00532b88', fontsize=fontsize)
ax.set_xlabel('Latitude [°N]', fontsize=fontsize)
ax.set_ylabel('Depth [m]', fontsize=fontsize)

ax = axs[0, 1]
ax.set_facecolor('gray')
p = da_bs1.plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
da_bs1.plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['k'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs1', fontsize=fontsize)
ax.set_xlabel('', fontsize=fontsize)
ax.set_ylabel('')

axs[1, 0].axis('off')

ax = axs[1, 1]
ax.set_facecolor('gray')
p = da_bs2.plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
da_bs2.plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['k'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs2', fontsize=fontsize)
ax.set_xlabel('Latitude [°N]', fontsize=fontsize)
ax.set_ylabel('')

pos0 = axs[1, 0].get_position()
pos1 = axs[1, 1].get_position()
cax = fig.add_axes([pos0.x0, pos0.y0 - 0.08, pos1.x1 - pos0.x0, 0.03])
fig.colorbar(p, cax=cax, orientation='horizontal', label='Temperature [°C]', extend='both')

ax = axs[0, 2]
ax.set_facecolor('gray')
vmax = 1.5
vmin = -vmax
cmap = cm.cm.balance

p = (da_bs1 - da).plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
(da_bs1 - da).plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['gray'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs1 minus test3v2-00532b88', fontsize=fontsize)
ax.set_xlabel('', fontsize=fontsize)
ax.set_ylabel('')

pos = ax.get_position()
cax = fig.add_axes([pos.x1 + 0.1, pos.y0 + 0.05, 0.015, pos.y1 - pos.y0])
fig.colorbar(p, cax = cax, orientation='vertical', label='Temperature [°C]', extend='both')

ax = axs[1, 2]
ax.set_facecolor('gray')
vmax = 1.5
vmin = -vmax
cmap = cm.cm.balance

p = (da_bs2 - da).plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
(da_bs2 - da).plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['gray'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs2 minus test3v2-00532b88', fontsize=fontsize)
ax.set_xlabel('Latitude [°N]', fontsize=fontsize)
ax.set_ylabel('')

pos = ax.get_position()
cax = fig.add_axes([pos.x1 + 0.1, pos.y0 + 0.05, 0.015, pos.y1 - pos.y0])
fig.colorbar(p, cax = cax, orientation='vertical', label='Temperature [°C]', extend='neither')

for ax in fig.get_axes():
    ax.label_outer()
plt.tight_layout(rect=[0, 0.1, 1, 1])

In [ ]:
var = 'so'
ds = om3vars_mean['Global'][var]

da = ds[list(ds)[0]]
da_bs1 = ds[list(ds)[1]]
da_bs2 = ds[list(ds)[2]]

dpi = 300
fontsize = 12
fig, axs = plt.subplots(2, 3, figsize=(25, 11), sharex=True, sharey=True)
fig.suptitle(f'Global zonal mean salinity, {time_start} - {time_end} mean', fontsize=fontsize + 2)

vmax = 35
vmin = 31
cmap = cm.cm.haline

ax = axs[0, 0]
ax.set_facecolor('gray')
p = da.plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
da.plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['k'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('test3v2-00532b88', fontsize=fontsize)
ax.set_xlabel('Latitude [°N]', fontsize=fontsize)
ax.set_ylabel('Depth [m]', fontsize=fontsize)

ax = axs[0, 1]
ax.set_facecolor('gray')
p = da_bs1.plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
da_bs1.plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['k'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs1', fontsize=fontsize)
ax.set_xlabel('', fontsize=fontsize)
ax.set_ylabel('')

axs[1, 0].axis('off')

ax = axs[1, 1]
ax.set_facecolor('gray')
p = da_bs2.plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
da_bs2.plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['k'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs2', fontsize=fontsize)
ax.set_xlabel('Latitude [°N]', fontsize=fontsize)
ax.set_ylabel('')

pos0 = axs[1, 0].get_position()
pos1 = axs[1, 1].get_position()
cax = fig.add_axes([pos0.x0, pos0.y0 - 0.08, pos1.x1 - pos0.x0, 0.03])
fig.colorbar(p, cax=cax, orientation='horizontal', label='Salinity [g/kg]', extend='both')

ax = axs[0, 2]
ax.set_facecolor('gray')
vmax = 0.4
vmin = -vmax
cmap = cm.cm.balance

p = (da_bs1 - da).plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
(da_bs1 - da).plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['gray'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs1 minus test3v2-00532b88', fontsize=fontsize)
ax.set_xlabel('', fontsize=fontsize)
ax.set_ylabel('')

pos = ax.get_position()
cax = fig.add_axes([pos.x1 + 0.1, pos.y0 + 0.05, 0.015, pos.y1 - pos.y0])
fig.colorbar(p, cax = cax, orientation='vertical', label='Salinity [g/kg]', extend='both')

ax = axs[1, 2]
ax.set_facecolor('gray')
vmax = 0.4
vmin = -vmax
cmap = cm.cm.balance

p = (da_bs2 - da).plot.contourf(ax = ax, levels=101, vmin=vmin, vmax=vmax, cmap=cmap, add_colorbar=False)
(da_bs2 - da).plot.contour(ax = ax, levels=25, vmin=vmin, vmax=vmax, add_colorbar=False, colors=['gray'], linewidths=[0.2], linestyles=['-'])
ax.set_ylim([0, None])
ax.invert_yaxis()
ax.set_title('bs2 minus test3v2-00532b88', fontsize=fontsize)
ax.set_xlabel('Latitude [°N]', fontsize=fontsize)
ax.set_ylabel('')

pos = ax.get_position()
cax = fig.add_axes([pos.x1 + 0.1, pos.y0 + 0.05, 0.015, pos.y1 - pos.y0])
fig.colorbar(p, cax = cax, orientation='vertical', label='Salinity [g/kg]', extend='neither')

for ax in fig.get_axes():
    ax.label_outer()
plt.tight_layout(rect=[0, 0.1, 1, 1])

In [ ]:
client.close()